# สตูดิโอพากเสียงบน Google Colab (GPU ฟรี)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cha12lotteZ/dub-studio/blob/main/notebooks/dub_studio_colab.ipynb)

ใช้แอปเต็มรูปแบบ รวมการแปลงเป็นเสียงคุณด้วยโมเดล RVC บน GPU ฟรีของ Colab

1. เมนู **Runtime → Change runtime type → T4 GPU**
2. กด **Runtime → Run all** แล้วอนุญาตให้เข้าถึง Google Drive (งานทั้งหมดจะเก็บไว้ใน Drive โฟลเดอร์ `dub-studio-data`)
3. รอประมาณ 5–10 นาทีในครั้งแรก แล้วกดลิงก์ที่ขึ้นในช่องสุดท้าย

Colab ฟรีปิดเองเมื่อไม่ได้ใช้งานสักพัก หรือใช้ต่อเนื่องนานประมาณ 12 ชั่วโมง งานที่บันทึกใน Drive ไม่หาย เปิดโน้ตบุ๊กนี้ใหม่ได้เลย

In [ ]:
# 1) เชื่อม Google Drive เพื่อเก็บโปรเจกต์ เทคเสียง และโมเดล
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/dub-studio-data"
import os
os.makedirs(DATA, exist_ok=True)


In [ ]:
# 2) ติดตั้งแอป (ประมาณ 3–5 นาที)
BRANCH = "main"
!rm -rf /content/dub-studio
!git clone -q --depth 1 --branch {BRANCH} https://github.com/Cha12lotteZ/dub-studio /content/dub-studio
%cd /content/dub-studio
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -e ".[ai]"
!python scripts/preload_models.py


In [ ]:
# 3) ติดตั้ง Applio สำหรับแปลงเสียงด้วยโมเดล RVC (ประมาณ 5 นาที ข้ามได้ถ้ายังไม่มีโมเดล)
INSTALL_APPLIO = True
APPLIO_VERSION = "3.7.2"
if INSTALL_APPLIO:
    !pip install -q uv
    !git clone -q --depth 1 --branch {APPLIO_VERSION} https://github.com/IAHispano/Applio.git /content/Applio
    !cd /content/Applio && uv venv -q .venv --python 3.12 && uv pip install -q --python .venv/bin/python -r requirements-engine.txt
    !cd /content/Applio && .venv/bin/python core.py prerequisites --models --no-exe
    os.environ["DUB_APPLIO_DIR"] = "/content/Applio"


In [ ]:
# 4) เปิดแอป แล้วกดลิงก์ที่ขึ้นด้านล่าง (ลิงก์ใช้ได้เฉพาะในเบราว์เซอร์ที่ล็อกอิน Google บัญชีนี้)
import subprocess, time, urllib.request
os.environ["DUB_DATA"] = DATA
server = subprocess.Popen(["python", "-m", "uvicorn", "dubstudio.main:app", "--host", "0.0.0.0", "--port", "8000"],
                          cwd="/content/dub-studio", stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8000/api/catalog", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/server.log").read()[-2000:])
    raise RuntimeError("แอปเปิดไม่ขึ้น ดู log ด้านบน")

from google.colab.output import eval_js
print("เปิดแอปที่:", eval_js("google.colab.kernel.proxyPort(8000)"))


### เทรนโมเดลเสียงของคุณ
ใช้โน้ตบุ๊ก [train_voice.ipynb](https://colab.research.google.com/github/Cha12lotteZ/dub-studio/blob/main/notebooks/train_voice.ipynb) แล้วอัปโหลด `.pth` และ `.index` เข้าแอปที่ช่อง **สอนเสียงของคุณ**